## Example 1: Basic WSGI Application with Request/Response

In [ ]:
from werkzeug.wrappers import Request, Response
from werkzeug.routing import Map, Rule
import werkzeug

# Example 1: Basic WSGI application with Request/Response wrappers
print("Basic WSGI application with Werkzeug:")

def simple_wsgi_app(environ, start_response):
    """Simple WSGI application using Werkzeug wrappers"""
    request = Request(environ)
    
    # Create response based on request path
    if request.path == '/':
        response_text = f"Hello Werkzeug! Method: {request.method}"
    elif request.path == '/info':
        response_text = f"Request info - Path: {request.path}, Args: {dict(request.args)}"
    else:
        response_text = f"Unknown path: {request.path}"
    
    response = Response(response_text, content_type='text/plain')
    return response(environ, start_response)

print("WSGI application defined")

# Test the WSGI app with mock environ
def test_wsgi_app(path='/', method='GET', query_string=''):
    """Test WSGI app with mock environment"""
    environ = {
        'REQUEST_METHOD': method,
        'PATH_INFO': path,
        'QUERY_STRING': query_string,
        'SERVER_NAME': 'localhost',
        'SERVER_PORT': '8000',
        'wsgi.version': (1, 0),
        'wsgi.url_scheme': 'http',
        'wsgi.input': None,
        'wsgi.errors': None,
        'wsgi.multithread': False,
        'wsgi.multiprocess': True,
        'wsgi.run_once': False
    }
    
    def start_response(status, headers):
        print(f"Response: {status}")
        for header in headers:
            print(f"Header: {header}")
    
    result = simple_wsgi_app(environ, start_response)
    response_body = b''.join(result).decode('utf-8')
    print(f"Body: {response_body}")
    return response_body

# Test different endpoints
print("\nTesting WSGI application:")
test_wsgi_app('/')
print()
test_wsgi_app('/info', query_string='test=123')
print()
test_wsgi_app('/unknown')

print(f"\nWerkzeug components:")
print(f"Request wrapper: {Request}")
print(f"Response wrapper: {Response}")
print("WSGI utilities for building web applications")

## Example 2: URL Routing and Dispatch System

In [ ]:
from werkzeug.routing import Map, Rule, Subdomain
from werkzeug.wrappers import Request, Response
from werkzeug.exceptions import NotFound, MethodNotAllowed

# Example 2: URL routing and dispatch system
print("URL routing and dispatch system:")

class RoutingApplication:
    """WSGI application with URL routing"""
    
    def __init__(self):
        # Define URL rules
        self.url_map = Map([
            Rule('/', endpoint='index'),
            Rule('/users', endpoint='users_list', methods=['GET']),
            Rule('/users/<int:user_id>', endpoint='user_detail', methods=['GET', 'POST']),
            Rule('/api/health', endpoint='health_check'),
            Rule('/api/data/<string:data_type>', endpoint='api_data'),
            # Subdomain routing example
            Subdomain('api', [
                Rule('/v1/status', endpoint='api_status'),
            ])
        ])
        
        # Endpoint handlers
        self.endpoints = {
            'index': self.index,
            'users_list': self.users_list,
            'user_detail': self.user_detail,
            'health_check': self.health_check,
            'api_data': self.api_data,
            'api_status': self.api_status
        }
    
    def index(self, request):
        """Home page handler"""
        return Response('Welcome to Werkzeug Routing Demo!', content_type='text/plain')
    
    def users_list(self, request):
        """Users list handler"""
        return Response('Users: Alice, Bob, Charlie', content_type='text/plain')
    
    def user_detail(self, request, user_id):
        """User detail handler"""
        if request.method == 'POST':
            return Response(f'Updated user {user_id}', content_type='text/plain')
        return Response(f'User {user_id} details', content_type='text/plain')
    
    def health_check(self, request):
        """Health check endpoint"""
        return Response('{"status": "healthy", "service": "werkzeug-demo"}', 
                       content_type='application/json')
    
    def api_data(self, request, data_type):
        """API data handler"""
        return Response(f'Data type: {data_type}, Query: {dict(request.args)}',
                       content_type='text/plain')
    
    def api_status(self, request):
        """API status handler (subdomain)"""
        return Response('API v1 Status: OK', content_type='text/plain')
    
    def dispatch_request(self, request):
        """Dispatch request to appropriate handler"""
        adapter = self.url_map.bind_to_environ(request.environ)
        
        try:
            endpoint, values = adapter.match()
            handler = self.endpoints.get(endpoint)
            
            if handler:
                return handler(request, **values)
            else:
                return Response('Handler not found', status=500)
                
        except NotFound:
            return Response('Not Found', status=404)
        except MethodNotAllowed:
            return Response('Method Not Allowed', status=405)
    
    def wsgi_app(self, environ, start_response):
        """Main WSGI application"""
        request = Request(environ)
        response = self.dispatch_request(request)
        return response(environ, start_response)

# Create routing application
app = RoutingApplication()
print("Routing application created")

# Test routing with various URLs
def test_routing(path, method='GET', query_string=''):
    """Test URL routing"""
    environ = {
        'REQUEST_METHOD': method,
        'PATH_INFO': path,
        'QUERY_STRING': query_string,
        'SERVER_NAME': 'localhost',
        'SERVER_PORT': '8000',
        'wsgi.version': (1, 0),
        'wsgi.url_scheme': 'http',
        'wsgi.input': None,
        'wsgi.errors': None,
        'wsgi.multithread': False,
        'wsgi.multiprocess': True,
        'wsgi.run_once': False
    }
    
    def start_response(status, headers):
        print(f"{path} -> {status}")
    
    request = Request(environ)
    response = app.dispatch_request(request)
    print(f"  Body: {response.get_data(as_text=True)}")

print("\nTesting URL routing:")
test_routing('/')
test_routing('/users')
test_routing('/users/123')
test_routing('/api/health')
test_routing('/api/data/json', query_string='filter=active')
test_routing('/nonexistent')

print(f"\nRouting features:")
print("- URL pattern matching with variables")
print("- HTTP method filtering")
print("- Subdomain routing support")
print("- Automatic 404/405 error handling")

## Example 3: Middleware and Exception Handling

In [ ]:
from werkzeug.middleware.shared_data import SharedDataMiddleware
from werkzeug.middleware.profiler import ProfilerMiddleware
from werkzeug.exceptions import HTTPException, BadRequest, InternalServerError
from werkzeug.wrappers import Request, Response
import json
import io

# Example 3: Middleware and exception handling
print("Middleware and exception handling:")

class CustomMiddleware:
    """Custom middleware for request/response processing"""
    
    def __init__(self, app):
        self.app = app
        self.request_count = 0
    
    def __call__(self, environ, start_response):
        # Pre-processing
        self.request_count += 1
        request = Request(environ)
        
        # Add custom headers
        def custom_start_response(status, headers):
            headers.append(('X-Request-Count', str(self.request_count)))
            headers.append(('X-Custom-Middleware', 'Werkzeug-Demo'))
            return start_response(status, headers)
        
        # Log request
        print(f"Request #{self.request_count}: {request.method} {request.path}")
        
        try:
            # Call the wrapped application
            return self.app(environ, custom_start_response)
        except Exception as e:
            # Exception handling
            print(f"Middleware caught exception: {e}")
            response = Response('Internal Server Error', status=500)
            return response(environ, custom_start_response)

class ExceptionHandlingApp:
    """Application demonstrating exception handling"""
    
    def __init__(self):
        self.data = {"users": [{"id": 1, "name": "Alice"}, {"id": 2, "name": "Bob"}]}
    
    def handle_request(self, request):
        """Main request handler with various exceptions"""
        
        if request.path == '/':
            return Response('Exception handling demo', content_type='text/plain')
        
        elif request.path == '/error/400':
            # Raise BadRequest
            raise BadRequest('Invalid request parameters')
        
        elif request.path == '/error/500':
            # Raise internal server error
            raise InternalServerError('Database connection failed')
        
        elif request.path == '/error/custom':
            # Raise custom exception
            raise ValueError('Custom application error')
        
        elif request.path == '/api/users':
            # JSON API endpoint
            if request.method == 'POST':
                try:
                    data = request.get_json(force=True)
                    if not data or 'name' not in data:
                        raise BadRequest('Missing required field: name')
                    
                    new_user = {"id": len(self.data["users"]) + 1, "name": data["name"]}
                    self.data["users"].append(new_user)
                    
                    return Response(json.dumps(new_user), 
                                  content_type='application/json', status=201)
                except json.JSONDecodeError:
                    raise BadRequest('Invalid JSON in request body')
            else:
                return Response(json.dumps(self.data), content_type='application/json')
        
        else:
            # 404 for unknown paths
            from werkzeug.exceptions import NotFound
            raise NotFound('Page not found')
    
    def wsgi_app(self, environ, start_response):
        """WSGI application with exception handling"""
        request = Request(environ)
        
        try:
            response = self.handle_request(request)
            return response(environ, start_response)
            
        except HTTPException as e:
            # Handle Werkzeug HTTP exceptions
            print(f"HTTP Exception: {e.code} - {e.description}")
            return e(environ, start_response)
            
        except Exception as e:
            # Handle other exceptions
            print(f"Unhandled exception: {e}")
            response = Response('Internal Server Error', status=500)
            return response(environ, start_response)

# Create application with middleware
base_app = ExceptionHandlingApp()
wrapped_app = CustomMiddleware(base_app.wsgi_app)

print("Application with middleware created")

# Test exception handling
def test_exception_handling(path, method='GET', json_data=None):
    """Test exception handling"""
    environ = {
        'REQUEST_METHOD': method,
        'PATH_INFO': path,
        'QUERY_STRING': '',
        'SERVER_NAME': 'localhost',
        'SERVER_PORT': '8000',
        'CONTENT_TYPE': 'application/json' if json_data else '',
        'wsgi.version': (1, 0),
        'wsgi.url_scheme': 'http',
        'wsgi.input': io.BytesIO(json.dumps(json_data).encode() if json_data else b''),
        'wsgi.errors': None,
        'wsgi.multithread': False,
        'wsgi.multiprocess': True,
        'wsgi.run_once': False
    }
    
    response_data = []
    def start_response(status, headers):
        print(f"  Response: {status}")
        for name, value in headers:
            if name.startswith('X-'):
                print(f"  Custom header: {name}: {value}")
    
    try:
        result = wrapped_app(environ, start_response)
        body = b''.join(result).decode('utf-8')
        print(f"  Body: {body[:100]}...")
    except Exception as e:
        print(f"  Exception: {e}")

print("\nTesting exception handling:")
test_exception_handling('/')
test_exception_handling('/error/400')
test_exception_handling('/error/500') 
test_exception_handling('/api/users')
test_exception_handling('/api/users', 'POST', {'name': 'Charlie'})
test_exception_handling('/nonexistent')

print(f"\nMiddleware features:")
print("- Request/response preprocessing")
print("- Custom header injection")
print("- Exception interception and handling")
print("- Request counting and logging")

## Example 4: File Handling and Security Utilities

In [ ]:
from werkzeug.utils import secure_filename, redirect
from werkzeug.datastructures import FileStorage
from werkzeug.http import parse_options_header
from werkzeug.wrappers import Request, Response
from werkzeug.exceptions import BadRequest
import io
import tempfile
import os

# Example 4: File handling and security utilities
print("File handling and security utilities:")

class FileHandlingApp:
    """Application demonstrating file uploads and security"""
    
    def __init__(self):
        self.upload_folder = tempfile.mkdtemp()
        self.allowed_extensions = {'txt', 'pdf', 'png', 'jpg', 'jpeg', 'gif'}
        print(f"Upload folder: {self.upload_folder}")
    
    def allowed_file(self, filename):
        """Check if file extension is allowed"""
        return ('.' in filename and 
                filename.rsplit('.', 1)[1].lower() in self.allowed_extensions)
    
    def handle_file_upload(self, request):
        """Handle file upload with security checks"""
        if request.method != 'POST':
            return Response('File upload form\n<form method="POST" enctype="multipart/form-data">\n'
                          '<input type="file" name="file">\n<input type="submit" value="Upload">\n</form>',
                          content_type='text/html')
        
        # Parse form data using Request.files
        try:
            form = request.form
            files = request.files
            
            if 'file' not in files:
                raise BadRequest('No file provided')
            
            file = files['file']
            
            # Check if file is selected
            if file.filename == '':
                raise BadRequest('No file selected')
            
            # Security check: secure filename
            if file and self.allowed_file(file.filename):
                filename = secure_filename(file.filename)
                
                # Additional security: limit file size (simulate)
                file_content = file.read()
                if len(file_content) > 1024 * 1024:  # 1MB limit
                    raise BadRequest('File too large (max 1MB)')
                
                # Save file
                filepath = os.path.join(self.upload_folder, filename)
                with open(filepath, 'wb') as f:
                    f.write(file_content)
                
                return Response(f'File {filename} uploaded successfully!\nPath: {filepath}',
                              content_type='text/plain')
            else:
                raise BadRequest('File type not allowed')
                
        except Exception as e:
            return Response(f'Upload error: {e}', status=400)
    
    def handle_redirect_example(self, request):
        """Demonstrate URL redirection"""
        if request.args.get('redirect') == 'home':
            return redirect('/')
        elif request.args.get('redirect') == 'external':
            # External redirect (be careful with user input!)
            url = request.args.get('url', 'https://example.com')
            return redirect(url)
        else:
            return Response('Redirect examples:\n'
                          '?redirect=home - redirect to home\n'
                          '?redirect=external&url=https://example.com - external redirect',
                          content_type='text/plain')
    
    def handle_security_headers(self, request):
        """Demonstrate security headers"""
        response = Response('Security headers example', content_type='text/plain')
        
        # Add security headers
        response.headers['X-Content-Type-Options'] = 'nosniff'
        response.headers['X-Frame-Options'] = 'DENY'
        response.headers['X-XSS-Protection'] = '1; mode=block'
        response.headers['Strict-Transport-Security'] = 'max-age=31536000; includeSubDomains'
        
        return response
    
    def wsgi_app(self, environ, start_response):
        """Main WSGI application"""
        request = Request(environ)
        
        if request.path == '/upload':
            response = self.handle_file_upload(request)
        elif request.path == '/redirect':
            response = self.handle_redirect_example(request)
        elif request.path == '/security':
            response = self.handle_security_headers(request)
        elif request.path == '/':
            response = Response(
                'File handling demo:\n'
                '/upload - file upload\n'
                '/redirect - redirection examples\n'
                '/security - security headers\n'
                '/filename/test file.txt - filename sanitization',
                content_type='text/plain'
            )
        elif request.path.startswith('/filename/'):
            # Demonstrate secure_filename utility
            filename = request.path.replace('/filename/', '')
            safe_filename = secure_filename(filename)
            response = Response(f'Original: {filename}\nSecure: {safe_filename}',
                              content_type='text/plain')
        else:
            response = Response('Not found', status=404)
        
        return response(environ, start_response)

# Create file handling application
file_app = FileHandlingApp()
print("File handling application created")

# Test secure_filename utility
def test_secure_filename():
    """Test secure_filename with various inputs"""
    test_filenames = [
        'normal_file.txt',
        '../../../etc/passwd',
        'file with spaces.pdf',
        'file$with#special@chars.jpg',
        'file..with..dots.png',
        'файл.txt',  # Unicode filename
        'CON.txt',   # Windows reserved name
        ''           # Empty filename
    ]
    
    print("\nTesting secure_filename:")
    for filename in test_filenames:
        safe = secure_filename(filename) 
        print(f"'{filename}' -> '{safe}'")

test_secure_filename()

# Test file handling endpoints
def test_file_endpoints():
    """Test file handling endpoints"""
    
    def test_endpoint(path, method='GET'):
        environ = {
            'REQUEST_METHOD': method,
            'PATH_INFO': path,
            'QUERY_STRING': '',
            'SERVER_NAME': 'localhost',
            'SERVER_PORT': '8000',
            'wsgi.version': (1, 0),
            'wsgi.url_scheme': 'http',
            'wsgi.input': io.BytesIO(),
            'wsgi.errors': None,
            'wsgi.multithread': False,
            'wsgi.multiprocess': True,
            'wsgi.run_once': False
        }
        
        def start_response(status, headers):
            print(f"{path} -> {status}")
            # Show security headers
            for name, value in headers:
                if name.startswith('X-') or 'Security' in name:
                    print(f"  {name}: {value}")
        
        request = Request(environ)
        response = file_app.handle_security_headers(request) if path == '/security' else Response('Test')
        result = response(environ, start_response)
        return b''.join(result).decode('utf-8')
    
    print(f"\nTesting security headers:")
    test_endpoint('/security')

test_file_endpoints()

# Test parse_options_header
print(f"\nTesting parse_options_header:")
content_type_examples = [
    'text/html; charset=utf-8',
    'application/json',
    'multipart/form-data; boundary=----WebKitFormBoundary',
    'text/plain; charset=iso-8859-1; format=flowed'
]

for content_type in content_type_examples:
    value, options = parse_options_header(content_type)
    print(f"'{content_type}'")
    print(f"  Value: {value}")
    print(f"  Options: {options}")

print(f"\nSecurity features:")
features = [
    "Secure filename sanitization (prevents directory traversal)",
    "File type validation by extension",
    "File size limiting",
    "Security headers injection (XSS, clickjacking protection)",
    "Safe URL redirection",
    "Content-Type parsing with options"
]

for feature in features:
    print(f"  ✓ {feature}")

print(f"\nWerkzeug provides comprehensive security utilities")
print("Essential for building secure web applications")

## Example 5: Production Patterns and Development Server

In [ ]:
from werkzeug.serving import make_server
from werkzeug.wrappers import Request, Response
from werkzeug.routing import Map, Rule
from werkzeug.middleware.dispatcher import DispatcherMiddleware
import threading
import time
import json

# Example 5: Production patterns and development server
print("Production patterns and development server:")

class ProductionWebApp:
    """Production-ready web application using Werkzeug"""
    
    def __init__(self):
        self.url_map = Map([
            Rule('/', endpoint='index'),
            Rule('/api/status', endpoint='health'),
            Rule('/api/users/<int:user_id>', endpoint='user_api'),
            Rule('/api/metrics', endpoint='metrics'),
            Rule('/static/<path:filename>', endpoint='static_files')
        ])
        
        self.users_db = {
            1: {"id": 1, "name": "Alice", "email": "alice@example.com"},
            2: {"id": 2, "name": "Bob", "email": "bob@example.com"}
        }
        
        self.metrics = {
            "requests_total": 0,
            "requests_by_endpoint": {},
            "start_time": time.time(),
            "errors": 0
        }
    
    def dispatch_request(self, request):
        """Dispatch requests to appropriate handlers"""
        adapter = self.url_map.bind_to_environ(request.environ)
        
        # Update metrics
        self.metrics["requests_total"] += 1
        
        try:
            endpoint, values = adapter.match()
            self.metrics["requests_by_endpoint"][endpoint] = \
                self.metrics["requests_by_endpoint"].get(endpoint, 0) + 1
            
            # Route to handler
            if endpoint == 'index':
                return self.handle_index(request)
            elif endpoint == 'health':
                return self.handle_health(request)
            elif endpoint == 'user_api':
                return self.handle_user_api(request, values['user_id'])
            elif endpoint == 'metrics':
                return self.handle_metrics(request)
            elif endpoint == 'static_files':
                return self.handle_static(request, values['filename'])
            else:
                return Response('Handler not implemented', status=500)
                
        except Exception as e:
            self.metrics["errors"] += 1
            return self.handle_error(request, e)
    
    def handle_index(self, request):
        """Home page handler"""
        return Response(
            'Werkzeug Production App\n'
            'Available endpoints:\n'
            '  /api/status - health check\n'
            '  /api/users/1 - user API\n'
            '  /api/metrics - application metrics\n'
            '  /static/style.css - static files',
            content_type='text/plain'
        )
    
    def handle_health(self, request):
        """Health check endpoint"""
        uptime = time.time() - self.metrics["start_time"]
        health_data = {
            "status": "healthy",
            "uptime_seconds": round(uptime, 2),
            "version": "1.0.0",
            "timestamp": time.time()
        }
        
        return Response(
            json.dumps(health_data, indent=2),
            content_type='application/json'
        )
    
    def handle_user_api(self, request, user_id):
        """User API endpoint"""
        if request.method == 'GET':
            user = self.users_db.get(user_id)
            if user:
                return Response(
                    json.dumps(user, indent=2),
                    content_type='application/json'
                )
            else:
                return Response(
                    json.dumps({"error": "User not found"}),
                    content_type='application/json',
                    status=404
                )
        
        elif request.method == 'PUT':
            # Update user
            try:
                data = request.get_json(force=True)
                if user_id in self.users_db:
                    self.users_db[user_id].update(data)
                    return Response(
                        json.dumps(self.users_db[user_id], indent=2),
                        content_type='application/json'
                    )
                else:
                    return Response(
                        json.dumps({"error": "User not found"}),
                        content_type='application/json',
                        status=404
                    )
            except Exception as e:
                return Response(
                    json.dumps({"error": str(e)}),
                    content_type='application/json',
                    status=400
                )
    
    def handle_metrics(self, request):
        """Metrics endpoint"""
        return Response(
            json.dumps(self.metrics, indent=2),
            content_type='application/json'
        )
    
    def handle_static(self, request, filename):
        """Static file handler (simplified)"""
        # In production, use proper static file serving
        if filename == 'style.css':
            css_content = """
            body { font-family: Arial, sans-serif; margin: 20px; }
            h1 { color: #333; }
            .api-response { background: #f5f5f5; padding: 10px; }
            """
            return Response(css_content, content_type='text/css')
        else:
            return Response('Static file not found', status=404)
    
    def handle_error(self, request, error):
        """Error handler"""
        print(f"Application error: {error}")
        return Response(
            json.dumps({"error": "Internal server error"}),
            content_type='application/json',
            status=500
        )
    
    def wsgi_app(self, environ, start_response):
        """Main WSGI application entry point"""
        request = Request(environ)
        response = self.dispatch_request(request)
        return response(environ, start_response)

# Create production application
prod_app = ProductionWebApp()
print("Production application created")

# Demonstrate application composition with DispatcherMiddleware
def create_admin_app():
    """Simple admin application"""
    def admin_wsgi(environ, start_response):
        request = Request(environ)
        response = Response(
            f'Admin Panel\nPath: {request.path}\nMethod: {request.method}',
            content_type='text/plain'
        )
        return response(environ, start_response)
    return admin_wsgi

admin_app = create_admin_app()

# Combine applications using DispatcherMiddleware
combined_app = DispatcherMiddleware(prod_app.wsgi_app, {
    '/admin': admin_app
})

print("Combined application with admin panel created")

# Test the production application
def test_production_app():
    """Test production application features"""
    
    def make_request(path, method='GET'):
        environ = {
            'REQUEST_METHOD': method,
            'PATH_INFO': path,
            'QUERY_STRING': '',
            'SERVER_NAME': 'localhost',
            'SERVER_PORT': '8000',
            'wsgi.version': (1, 0),
            'wsgi.url_scheme': 'http',
            'wsgi.input': None,
            'wsgi.errors': None,
            'wsgi.multithread': False,
            'wsgi.multiprocess': True,
            'wsgi.run_once': False
        }
        
        def start_response(status, headers):
            print(f"{path} -> {status}")
        
        result = combined_app(environ, start_response)
        body = b''.join(result).decode('utf-8')
        print(f"  Response: {body[:100]}...")
        return body
    
    print(f"\nTesting production application:")
    make_request('/')
    make_request('/api/status')
    make_request('/api/users/1')
    make_request('/api/metrics')
    make_request('/admin/dashboard')

test_production_app()

# Demonstrate development server setup (conceptual)
print(f"\nDevelopment server setup:")
print("# Create development server with Werkzeug")
print("server = make_server('localhost', 8000, combined_app)")
print("# server.serve_forever()  # Would run the server")

print(f"\nProduction features:")
print("- Structured request routing")
print("- JSON API endpoints")
print("- Health checks and metrics")
print("- Error handling and logging")
print("- Application composition")
print("- Static file serving")
print("- Development server integration")